## What's the wmo with the most active time and with the largest distance covered ?

In [ ]:
import pandas as pd
import numpy as np
# LOAD INDEX
df=pd.read_csv('/scale/reference/argo/gdac/ar_index_global_prof.txt.gz',sep=',', index_col=None, 
                  header=0, skiprows=8,parse_dates=[1, 7])
# MAKE THE DATE PANDAS DATETIME FOR FUTUR USE
df['date']=pd.to_datetime(df.date)
# GENERATE FLOAT NUMBER FROM FILE
df['FloatNumber'] = df.file.apply(lambda x: int(x.split('/')[1]))
#
df.head(5)

,file,date,latitude,longitude,ocean,profiler_type,institution,date_update,FloatNumber
0,aoml/13857/profiles/D13857_001.nc,1997-07-29 20:03:00,0.267,-16.032,A,845,AO,2026-02-20 14:35:29,13857
1,aoml/13857/profiles/D13857_002.nc,1997-08-09 19:21:12,0.072,-17.659,A,845,AO,2026-02-20 14:35:29,13857
2,aoml/13857/profiles/D13857_003.nc,1997-08-20 18:45:45,0.543,-19.622,A,845,AO,2026-02-20 14:35:30,13857
3,aoml/13857/profiles/D13857_004.nc,1997-08-31 19:39:05,1.256,-20.521,A,845,AO,2026-02-20 14:35:30,13857
4,aoml/13857/profiles/D13857_005.nc,1997-09-11 18:58:08,0.720,-20.768,A,845,AO,2026-02-20 14:35:30,13857


In [ ]:
ds = df.to_xarray()
ds

<xarray.Dataset> Size: 269MB
Dimensions:        (index: 3366778)
Coordinates:
  * index          (index) int64 27MB 0 1 2 3 ... 3366775 3366776 3366777
Data variables:
    file           (index) object 27MB 'aoml/13857/profiles/D13857_001.nc' .....
    date           (index) datetime64[ns] 27MB 1997-07-29T20:03:00 ... 2013-0...
    latitude       (index) float64 27MB 0.267 0.072 0.543 ... 27.69 27.89 27.93
    longitude      (index) float64 27MB -16.03 -17.66 -19.62 ... 138.5 138.1
    ocean          (index) object 27MB 'A' 'A' 'A' 'A' 'A' ... 'P' 'P' 'P' 'P'
    profiler_type  (index) int64 27MB 845 845 845 845 845 ... 841 841 841 841
    institution    (index) object 27MB 'AO' 'AO' 'AO' 'AO' ... 'NM' 'NM' 'NM'
    date_update    (index) datetime64[ns] 27MB 2026-02-20T14:35:29 ... 2013-0...
    FloatNumber    (index) int64 27MB 13857 13857 13857 ... 2901633 2901633

In [ ]:
# FUNC TO CALCULATE ACTIVE AGE
def stretch(ds):    
    return(ds['date'].max()-ds['date'].min())

In [ ]:
# GROUP BY FLOAT NUMBER AND APPLY THE FUNCTION PER GROUP
time_stretch = ds.groupby('FloatNumber').map(stretch)
time_stretch = time_stretch.dropna('FloatNumber')

<xarray.DataArray 'date' ()> Size: 8B
array(509311119000000000, dtype='timedelta64[ns]')
Coordinates:
    FloatNumber  int64 8B 5901055

In [ ]:
#GET THE MOST ACTIVE FLOATS
np.argpartition(time_stretch.values,-5)[-5:]

array([10519, 11088, 10881,  1445, 10878])

In [ ]:
champ = time_stretch.isel(FloatNumber=10881)
champ.values.astype('timedelta64[Y]')

numpy.timedelta64(15,'Y')

In [ ]:
champ

<xarray.DataArray 'date' ()> Size: 8B
array(485533602000000000, dtype='timedelta64[ns]')
Coordinates:
    FloatNumber  int64 8B 5901058

## LONGEST TRAJECTORY

In [ ]:
import xarray as xr
import shapely
from cartopy import geodesic

In [ ]:
# USE SHAPELY AND GEODESIC TO CALCULATE DISTANCE ALONG TRAJECTORY
def distance(dsf):    
    try:
        dsf = dsf.sortby('date')
        dsf = dsf.dropna('index')        
        path2 = shapely.geometry.LineString(np.vstack([dsf['longitude'].values,dsf['latitude'].values]).transpose())    
        return xr.DataArray(geodesic.Geodesic().geometry_length(path2)/1000)
    except:
        return xr.DataArray(0.0)

In [ ]:
# TEST
dsf = ds.where(ds['FloatNumber']==5901055,drop=True)
dsf = dsf.sortby('date')
dsf = dsf.dropna('index')

In [ ]:
path2 = shapely.geometry.LineString(np.vstack([dsf['longitude'].values,dsf['latitude'].values]).transpose())

In [ ]:
geodesic.Geodesic().geometry_length(path2)/1000

16632.599949438307

In [ ]:
# MAP FUNCTION TO ALL FLOATS
distances_f = ds.groupby('FloatNumber').map(distance)
distances_f

<xarray.DataArray (FloatNumber: 20321)> Size: 163kB
array([ 7242.41231165,  2962.96821977, 10054.33151249, ...,
          33.54111379,   102.43012504,   115.33067743])
Coordinates:
  * FloatNumber  (FloatNumber) int64 163kB 13857 13858 13859 ... 7902428 7902429

In [ ]:
# GET THE LONGEST DISTANCE FLOATS
np.argpartition(distances_f.values,-5)[-5:]

array([ 7842, 14368, 10957,  8957,  8889])

In [ ]:
distances_f.isel(FloatNumber=14368)

<xarray.DataArray ()> Size: 8B
array(46759.03706551)
Coordinates:
    FloatNumber  int64 8B 5904952